# Final test-set scores

This notebook scores the crater detectors on the **test set**: U-Net run 2, Faster R-CNN, DINOv3 + head,
and the basic MLP and CNN baselines.
Nothing is trained here. Each model uses the cutoff that was already chosen on the validation set.

## 1. Imports and setup

Load the libraries and pick the device (GPU if there is one).

In [8]:
import json
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from scipy import ndimage
from sklearn.model_selection import train_test_split
from torchvision.ops import box_iou
from tqdm import tqdm

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Using device:", device)

Using device: cuda


## 2. Data split

The same split as every other notebook: 80% train, then the remaining 20% split in half into validation and test.
We only use `test_ids` here.

In [9]:
images_path = Path("dataset/images")
data = json.loads(Path("dataset/gt_public.json").read_text())

image_ids = list(data.keys())
train_ids, rest_ids = train_test_split(image_ids, test_size=0.2, random_state=0)
val_ids, test_ids = train_test_split(rest_ids, test_size=0.5, random_state=0)

print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

Train: 41470, Val: 5184, Test: 5184


## 3. Shared scoring functions

These are the same functions used for the validation scores:

- `get_true_boxes`: turns the (x, y, w, h) labels into (x1, y1, x2, y2) boxes, like in 06-faster-rcnn.ipynb.
- `count_matches`: a predicted box finds a true crater if their box IoU is at least 0.5. Each true crater can be matched only once.
- `mask_to_boxes`: for the mask models, each connected group of crater pixels becomes one box.
- `get_scores`: precision, recall and F1, counted over all test images together.
- `make_mask`: draws the true crater mask (copied from 05-unet.ipynb). Needed for pixel IoU.
- `save_scores`: writes the scores to `test_scores.json` in the model's run folder.

Pixel IoU is counted over all test images together: total overlap pixels divided by total union pixels.

In [10]:
def get_true_boxes(image_id):
    boxes = [[x, y, x + w, y + h] for x, y, w, h in data[image_id] if w > 0 and h > 0]
    return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def count_matches(pred_boxes, true_boxes, iou_threshold=0.5):
    if len(pred_boxes) == 0 or len(true_boxes) == 0:
        return 0

    ious = box_iou(pred_boxes, true_boxes)
    matches = 0
    for i in range(len(pred_boxes)):
        j = ious[i].argmax()
        if ious[i, j] >= iou_threshold:
            matches += 1
            ious[:, j] = -1  # this real crater is taken
    return matches


def mask_to_boxes(pred_mask):
    labeled, n = ndimage.label(pred_mask)
    boxes = []
    for ys, xs in ndimage.find_objects(labeled):
        boxes.append([xs.start, ys.start, xs.stop, ys.stop])
    return torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)


def get_scores(tp, n_pred, n_true):
    precision = tp / n_pred if n_pred > 0 else 0.0
    recall = tp / n_true
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1


def make_mask(boxes):
    mask = np.zeros((256, 256), dtype=np.uint8)
    yy, xx = np.mgrid[0:256, 0:256]

    for x, y, w, h in boxes:
        cx = x + w / 2
        cy = y + h / 2
        r = max(w, h) / 2
        inside = (xx + 0.5 - cx) ** 2 + (yy + 0.5 - cy) ** 2 <= r ** 2
        mask[inside] = 1

    return mask


def save_scores(run_dir, scores):
    (run_dir / "test_scores.json").write_text(json.dumps(scores, indent=2))
    for name, value in scores.items():
        print(f"{name}: {value:.4f}")

## 4. U-Net (run 2)

The model class is copied from 05-unet.ipynb. We load the saved weights and predict a crater mask for every test image.

- Per crater: keep pixels with probability > 0.9, turn the mask into boxes, and count matches.
- Pixel IoU: at cutoff 0.5 (same as the MLP and CNN) and at 0.9 (the U-Net's chosen cutoff).

In [11]:
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
    )


class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # shrinking half
        self.down1 = conv_block(1, 32)      # (32, 256, 256)
        self.down2 = conv_block(32, 64)     # (64, 128, 128)
        self.down3 = conv_block(64, 128)    # (128, 64, 64)
        self.bottom = conv_block(128, 256)  # (256, 32, 32)

        # growing half
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)  # (128, 64, 64)
        self.conv3 = conv_block(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)   # (64, 128, 128)
        self.conv2 = conv_block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)    # (32, 256, 256)
        self.conv1 = conv_block(64, 32)

        self.out = nn.Conv2d(32, 1, kernel_size=1)  # (1, 256, 256)

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(self.pool(d1))
        d3 = self.down3(self.pool(d2))
        b = self.bottom(self.pool(d3))

        u3 = self.conv3(torch.cat([self.up3(b), d3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u3), d2], dim=1))
        u1 = self.conv1(torch.cat([self.up1(u2), d1], dim=1))
        return self.out(u1)

In [16]:
unet_dir = Path("src/unet/02-cosine-lr")
unet_cutoff = 0.9
unet_pixel_cutoffs = [0.5, 0.8]

unet = UNet().to(device)
unet.load_state_dict(torch.load(unet_dir / "unet.pt", map_location=device))
unet.eval()


# same image loading as the CraterDataset in 05-unet.ipynb (grayscale, 0 to 1)
def load_gray_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("L")
    image = torch.tensor(np.array(image), dtype=torch.float32) / 255.0
    return image.unsqueeze(0)


tp = 0
n_pred = 0
n_true = 0
intersection = {c: 0 for c in unet_pixel_cutoffs}
union = {c: 0 for c in unet_pixel_cutoffs}

for start in tqdm(range(0, len(test_ids), 16), mininterval=1.0):
    batch_ids = test_ids[start:start + 16]
    images = torch.stack([load_gray_image(i) for i in batch_ids]).to(device)

    with torch.no_grad():
        probs = torch.sigmoid(unet(images)).squeeze(1).cpu().numpy()

    for prob, image_id in zip(probs, batch_ids):
        # per-crater matching
        true_boxes = get_true_boxes(image_id)
        pred_boxes = mask_to_boxes(prob > unet_cutoff)
        n_true += len(true_boxes)
        n_pred += len(pred_boxes)
        tp += count_matches(pred_boxes, true_boxes)

        # pixel IoU
        true_mask = make_mask(data[image_id]) == 1
        for c in unet_pixel_cutoffs:
            intersection[c] += ((prob > c) & true_mask).sum()
            union[c] += ((prob > c) | true_mask).sum()

unet_precision, unet_recall, unet_f1 = get_scores(tp, n_pred, n_true)
unet_iou = {c: float(intersection[c] / union[c]) for c in unet_pixel_cutoffs}

save_scores(unet_dir, {
    "cutoff": unet_cutoff,
    "precision": unet_precision,
    "recall": unet_recall,
    "f1": unet_f1,
    "pixel_iou_0.5": unet_iou[0.5],
    "pixel_iou_0.8": unet_iou[0.8],
})

100%|██████████| 324/324 [00:14<00:00, 22.58it/s]

cutoff: 0.9000
precision: 0.4811
recall: 0.7029
f1: 0.5712
pixel_iou_0.5: 0.4142
pixel_iou_0.8: 0.5115


## 5. Faster R-CNN

Built the same way as in 06-faster-rcnn.ipynb: ResNet-50 + FPN, `min_size=256`, and a new 2-class head.
Then we load the saved weights, keep predicted boxes with score > 0.8, and count matches.
Faster R-CNN predicts boxes, not pixels, so it has no pixel IoU.

In [6]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

frcnn_dir = Path("src/faster-rcnn/01-baseline")
frcnn_cutoff = 0.8

frcnn = fasterrcnn_resnet50_fpn(weights="DEFAULT", min_size=256)
in_features = frcnn.roi_heads.box_predictor.cls_score.in_features
frcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=2)

frcnn.load_state_dict(torch.load(frcnn_dir / "model.pt", map_location=device))
frcnn.to(device)
frcnn.eval()


# same image loading as the CraterDetectionDataset in 06-faster-rcnn.ipynb (RGB, 0 to 1)
def load_rgb_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("RGB")
    return torch.tensor(np.array(image), dtype=torch.float32).permute(2, 0, 1) / 255.0


tp = 0
n_pred = 0
n_true = 0

for start in tqdm(range(0, len(test_ids), 8), mininterval=1.0):
    batch_ids = test_ids[start:start + 8]
    images = [load_rgb_image(i).to(device) for i in batch_ids]

    with torch.no_grad():
        outputs = frcnn(images)

    for out, image_id in zip(outputs, batch_ids):
        true_boxes = get_true_boxes(image_id)
        pred_boxes = out["boxes"][out["scores"] > frcnn_cutoff].cpu()
        n_true += len(true_boxes)
        n_pred += len(pred_boxes)
        tp += count_matches(pred_boxes, true_boxes)

frcnn_precision, frcnn_recall, frcnn_f1 = get_scores(tp, n_pred, n_true)

save_scores(frcnn_dir, {
    "cutoff": frcnn_cutoff,
    "precision": frcnn_precision,
    "recall": frcnn_recall,
    "f1": frcnn_f1,
})

  0%|          | 0/648 [00:00<?, ?it/s]

  0%|          | 3/648 [00:01<04:01,  2.67it/s]

  1%|          | 7/648 [00:02<03:26,  3.10it/s]

  2%|▏         | 11/648 [00:03<03:16,  3.24it/s]

  2%|▏         | 15/648 [00:04<03:11,  3.31it/s]

  3%|▎         | 19/648 [00:05<03:08,  3.34it/s]

... (163 outputs left out)

 99%|█████████▉| 642/648 [03:22<00:02,  2.80it/s]

100%|█████████▉| 646/648 [03:24<00:00,  2.85it/s]

100%|██████████| 648/648 [03:24<00:00,  3.16it/s]

cutoff: 0.8000
precision: 0.7892
recall: 0.8047
f1: 0.7969


## 6. DINOv3 + head

The backbone, image preprocessing, feature extraction and head are copied from 07-dinov3.ipynb.
We extract the test features once with the frozen backbone and run the saved head.

- Per crater: keep pixels with probability > 0.9, turn the mask into boxes, and count matches.
- Pixel IoU: at cutoff 0.5 and at 0.9 (DINOv3's chosen cutoff).

DINOv3 is a gated model on Hugging Face, so you must be logged in once on this computer (`hf auth login` in a terminal).

In [7]:
from transformers import AutoModel

model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"
dino = AutoModel.from_pretrained(model_name).to(device)
dino.eval()

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_image(image_id):
    image = Image.open(images_path / f"{image_id}.png").convert("RGB")
    image = torch.tensor(np.array(image), dtype=torch.float32).permute(2, 0, 1) / 255
    image = (image - mean) / std
    return image


skip = 1 + dino.config.num_register_tokens

def get_features(ids, batch_size=32):
    all_features = []
    for start in tqdm(range(0, len(ids), batch_size), mininterval=1.0):
        batch_ids = ids[start:start + batch_size]
        images = torch.stack([load_image(i) for i in batch_ids]).to(device)

        with torch.no_grad():
            output = dino(pixel_values=images)

        patches = output.last_hidden_state[:, skip:, :]
        patches = patches.reshape(-1, 16, 16, 384).permute(0, 3, 1, 2)
        all_features.append(patches.half().cpu())

    return torch.cat(all_features)


test_features = get_features(test_ids)
print(test_features.shape)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1111 10:18:58.299000 24756 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 211/211 [00:00<00:00, 10116.92it/s]

  0%|          | 0/162 [00:00<?, ?it/s]

  4%|▎         | 6/162 [00:01<00:28,  5.53it/s]

... (21 outputs left out)

 95%|█████████▌| 154/162 [00:25<00:01,  6.17it/s]

 99%|█████████▉| 161/162 [00:26<00:00,  6.21it/s]

100%|██████████| 162/162 [00:26<00:00,  6.04it/s]

torch.Size([5184, 384, 16, 16])


In [8]:
class DinoHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.out = nn.Conv2d(384, 1, kernel_size=1)

    def forward(self, x):
        x = self.out(x)
        x = F.interpolate(x, size=(256, 256), mode="bilinear", align_corners=False)
        return x

In [9]:
dino_dir = Path("src/dinov3/01-baseline")
dino_cutoff = 0.9
dino_pixel_cutoffs = [0.5, 0.9]

head = DinoHead().to(device)
head.load_state_dict(torch.load(dino_dir / "model.pt", map_location=device))
head.eval()

tp = 0
n_pred = 0
n_true = 0
intersection = {c: 0 for c in dino_pixel_cutoffs}
union = {c: 0 for c in dino_pixel_cutoffs}

for start in tqdm(range(0, len(test_ids), 64), mininterval=1.0):
    batch_ids = test_ids[start:start + 64]

    with torch.no_grad():
        probs = torch.sigmoid(head(test_features[start:start + 64].float().to(device)))
    probs = probs.squeeze(1).cpu().numpy()

    for prob, image_id in zip(probs, batch_ids):
        # per-crater matching
        true_boxes = get_true_boxes(image_id)
        pred_boxes = mask_to_boxes(prob > dino_cutoff)
        n_true += len(true_boxes)
        n_pred += len(pred_boxes)
        tp += count_matches(pred_boxes, true_boxes)

        # pixel IoU
        true_mask = make_mask(data[image_id]) == 1
        for c in dino_pixel_cutoffs:
            intersection[c] += ((prob > c) & true_mask).sum()
            union[c] += ((prob > c) | true_mask).sum()

dino_precision, dino_recall, dino_f1 = get_scores(tp, n_pred, n_true)
dino_iou = {c: float(intersection[c] / union[c]) for c in dino_pixel_cutoffs}

save_scores(dino_dir, {
    "cutoff": dino_cutoff,
    "precision": dino_precision,
    "recall": dino_recall,
    "f1": dino_f1,
    "pixel_iou_0.5": dino_iou[0.5],
    "pixel_iou_0.9": dino_iou[0.9],
})

  0%|          | 0/81 [00:00<?, ?it/s]

 22%|██▏       | 18/81 [00:01<00:03, 17.54it/s]

 44%|████▍     | 36/81 [00:02<00:02, 17.33it/s]

 67%|██████▋   | 54/81 [00:03<00:01, 16.70it/s]

 88%|████████▊ | 71/81 [00:04<00:00, 16.76it/s]

100%|██████████| 81/81 [00:04<00:00, 16.96it/s]

cutoff: 0.9000
precision: 0.3061
recall: 0.1581
f1: 0.2085
pixel_iou_0.5: 0.4142
pixel_iou_0.9: 0.4848


## 7. Basic MLP and basic CNN

These two baselines were only checked with pixel IoU at cutoff 0.5 on validation, so we use cutoff 0.5 here too.
For each one we compute the same per-crater precision, recall and F1 as above, and the pixel IoU at 0.5.

In [10]:
def score_baseline(model, cutoff):
    tp = 0
    n_pred = 0
    n_true = 0
    intersection = 0
    union = 0

    for start in tqdm(range(0, len(test_ids), 16), mininterval=1.0):
        batch_ids = test_ids[start:start + 16]
        images = torch.stack([load_gray_image(i) for i in batch_ids]).to(device)

        with torch.no_grad():
            probs = torch.sigmoid(model(images)).squeeze(1).cpu().numpy()

        for prob, image_id in zip(probs, batch_ids):
            pred_mask = prob > cutoff

            # per-crater matching
            true_boxes = get_true_boxes(image_id)
            pred_boxes = mask_to_boxes(pred_mask)
            n_true += len(true_boxes)
            n_pred += len(pred_boxes)
            tp += count_matches(pred_boxes, true_boxes)

            # pixel IoU
            true_mask = make_mask(data[image_id]) == 1
            intersection += (pred_mask & true_mask).sum()
            union += (pred_mask | true_mask).sum()

    precision, recall, f1 = get_scores(tp, n_pred, n_true)
    pixel_iou = float(intersection / union)
    return precision, recall, f1, pixel_iou

### Basic MLP

The model class is copied from 03-mlp.ipynb.

In [11]:
class PatchMLP(nn.Module):
    def __init__(self, patch=16):
        super().__init__()
        self.patch = patch
        self.mlp = nn.Sequential(
            nn.Linear(patch * patch, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, patch * patch),
        )

    def forward(self, x):
        b, p = x.shape[0], self.patch
        n = 256 // p

        # cut each image into 16 x 16 patches, flatten each patch
        x = x.view(b, n, p, n, p).permute(0, 1, 3, 2, 4).reshape(b * n * n, p * p)
        x = self.mlp(x)
        # put the patches back together into a 256 x 256 map
        x = x.view(b, n, n, p, p).permute(0, 1, 3, 2, 4).reshape(b, 1, 256, 256)
        return x

In [12]:
mlp_dir = Path("src/mlp/01-baseline")
mlp_cutoff = 0.5

mlp = PatchMLP().to(device)
mlp.load_state_dict(torch.load(mlp_dir / "model.pt", map_location=device))
mlp.eval()

mlp_precision, mlp_recall, mlp_f1, mlp_iou = score_baseline(mlp, mlp_cutoff)

save_scores(mlp_dir, {
    "cutoff": mlp_cutoff,
    "precision": mlp_precision,
    "recall": mlp_recall,
    "f1": mlp_f1,
    "pixel_iou_0.5": mlp_iou,
})

  0%|          | 0/324 [00:00<?, ?it/s]

 10%|█         | 33/324 [00:01<00:08, 32.38it/s]

 21%|██        | 67/324 [00:02<00:07, 32.96it/s]

 31%|███       | 100/324 [00:03<00:06, 32.87it/s]

 41%|████      | 133/324 [00:04<00:05, 32.90it/s]

 52%|█████▏    | 167/324 [00:05<00:04, 33.05it/s]

... (2 outputs left out)

 83%|████████▎ | 268/324 [00:08<00:01, 32.22it/s]

 93%|█████████▎| 301/324 [00:09<00:00, 31.89it/s]

100%|██████████| 324/324 [00:09<00:00, 32.41it/s]

cutoff: 0.5000
precision: 0.0669
recall: 0.3163
f1: 0.1105
pixel_iou_0.5: 0.1758


### Basic CNN

The model class is copied from 04-cnn.ipynb (it uses the same `conv_block` as the U-Net, defined above).

In [13]:
class PlainCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            conv_block(1, 32),    # (32, 256, 256)
            conv_block(32, 64),   # (64, 256, 256)
            conv_block(64, 64),   # (64, 256, 256)
            conv_block(64, 32),   # (32, 256, 256)
            nn.Conv2d(32, 1, kernel_size=1),  # (1, 256, 256)
        )

    def forward(self, x):
        return self.layers(x)

In [14]:
cnn_dir = Path("src/cnn/01-baseline")
cnn_cutoff = 0.5

cnn = PlainCNN().to(device)
cnn.load_state_dict(torch.load(cnn_dir / "model.pt", map_location=device))
cnn.eval()

cnn_precision, cnn_recall, cnn_f1, cnn_iou = score_baseline(cnn, cnn_cutoff)

save_scores(cnn_dir, {
    "cutoff": cnn_cutoff,
    "precision": cnn_precision,
    "recall": cnn_recall,
    "f1": cnn_f1,
    "pixel_iou_0.5": cnn_iou,
})

  0%|          | 0/324 [00:00<?, ?it/s]

  4%|▎         | 12/324 [00:01<00:27, 11.40it/s]

  7%|▋         | 24/324 [00:02<00:26, 11.50it/s]

 11%|█         | 36/324 [00:03<00:24, 11.53it/s]

 15%|█▍        | 48/324 [00:04<00:23, 11.70it/s]

 19%|█▊        | 60/324 [00:05<00:22, 11.66it/s]

... (19 outputs left out)

 94%|█████████▎| 303/324 [00:25<00:01, 11.97it/s]

 97%|█████████▋| 315/324 [00:26<00:00, 11.97it/s]

100%|██████████| 324/324 [00:27<00:00, 11.88it/s]

cutoff: 0.5000
precision: 0.0312
recall: 0.6338
f1: 0.0594
pixel_iou_0.5: 0.2153


## 8. Final tables

**Per-crater scores.** Validation F1 values are from src/comparison/crater_scores.csv.
The MLP and CNN were never scored per crater on validation, so they show "-".

In [15]:
def show(value):
    if value is None:
        return "-"
    return f"{value:.4f}"


crater_rows = [
    ["MLP", mlp_cutoff, mlp_precision, mlp_recall, mlp_f1, None],
    ["CNN", cnn_cutoff, cnn_precision, cnn_recall, cnn_f1, None],
    ["U-Net 2", unet_cutoff, unet_precision, unet_recall, unet_f1, 0.5818],
    ["DINOv3", dino_cutoff, dino_precision, dino_recall, dino_f1, 0.2143],
    ["Faster R-CNN", frcnn_cutoff, frcnn_precision, frcnn_recall, frcnn_f1, 0.7954],
]

print(f"{'Model':<14}{'Cutoff':>8}{'Test P':>9}{'Test R':>9}{'Test F1':>9}{'Val F1':>9}")
for name, cutoff, precision, recall, f1, val_f1 in crater_rows:
    print(f"{name:<14}{cutoff:>8}{show(precision):>9}{show(recall):>9}{show(f1):>9}{show(val_f1):>9}")

Model           Cutoff   Test P   Test R  Test F1   Val F1
MLP                0.5   0.0669   0.3163   0.1105        -
CNN                0.5   0.0312   0.6338   0.0594        -
U-Net 2            0.9   0.4813   0.7028   0.5713   0.5818
DINOv3             0.9   0.3061   0.1581   0.2085   0.2143
Faster R-CNN       0.8   0.7892   0.8047   0.7969   0.7954


**Pixel IoU.** Test pixel IoU at cutoff 0.5 (the same cutoff for every model) and at 0.9 (the chosen cutoff of U-Net and DINOv3).
Validation IoU values are from src/comparison/pixel_iou.csv, at the cutoff written next to them.
Faster R-CNN is not in this table because it predicts boxes, not pixels.

In [16]:
pixel_rows = [
    ["MLP", mlp_iou, None, 0.1814, 0.5],
    ["CNN", cnn_iou, None, 0.2207, 0.5],
    ["U-Net 2", unet_iou[0.5], unet_iou[0.9], 0.5188, 0.8],
    ["DINOv3", dino_iou[0.5], dino_iou[0.9], 0.4975, 0.8],
]

print(f"{'Model':<10}{'Test IoU @0.5':>15}{'Test IoU @0.9':>15}{'Val IoU':>10}{'(cutoff)':>10}")
for name, iou_05, iou_09, val_iou, val_cutoff in pixel_rows:
    print(f"{name:<10}{show(iou_05):>15}{show(iou_09):>15}{show(val_iou):>10}{val_cutoff:>10}")

Model       Test IoU @0.5  Test IoU @0.9   Val IoU  (cutoff)
MLP                0.1758              -    0.1814       0.5
CNN                0.2153              -    0.2207       0.5
U-Net 2            0.4143         0.4880    0.5188       0.8
DINOv3             0.4142         0.4848    0.4975       0.8
